## 9.0 Introduction

Since Lecture 1 we have been building toward one question: given a dataset, how do we find the weights $x_1, x_2, \ldots, x_n$ that make a linear prediction model work as well as possible? In Lecture 7 we learned that matrix-vector multiplication lets us write all of those equations at once as $A\mathbf{x} = \mathbf{b}$. This lecture develops the tools to actually solve it.

We'll work through three ideas in order:

1. What it means to solve $A\mathbf{x} = \mathbf{b}$, and when a solution exists
2. How the matrix inverse $A^{-1}$ solves the square, full-rank case exactly
3. How the **least squares solution** $\hat{\mathbf{x}} = (A^TA)^{-1}A^T\mathbf{b}$ handles the over-determined case — the one that always arises with real data

By the end of this lecture you will be able to:

- Interpret $A\mathbf{x} = \mathbf{b}$ as asking whether $\mathbf{b}$ is reachable by linear combinations of the columns of $A$
- Define the column space $C(A)$ and matrix rank
- Use the matrix inverse to solve square, full-rank systems
- Derive and apply the least squares solution $\hat{\mathbf{x}} = (A^TA)^{-1}A^T\mathbf{b}$
- Recognize the squared error $\|\mathbf{b} - A\mathbf{x}\|^2$ as an objective function and explain why minimizing it is the central idea in supervised learning

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 9.1 $A\mathbf{x} = \mathbf{b}$, Column Space, and Rank

### The Equation

The central equation of this lecture is $A\mathbf{x} = \mathbf{b}$, where $A$ is an $m \times n$ matrix, $\mathbf{x} \in \mathbb{R}^n$ is unknown, and $\mathbf{b} \in \mathbb{R}^m$ is known. We are solving for $\mathbf{x}$.

As a concrete example, let:

$$A = \begin{bmatrix} 3 & 0 \\ 1 & 2 \\ 0 & 1 \end{bmatrix}, \qquad \mathbf{x} = \begin{bmatrix} x_1 \\ x_2 \end{bmatrix}, \qquad \mathbf{b} = \begin{bmatrix} 3 \\ 4 \\ 1.5 \end{bmatrix}$$

Written out row by row, $A\mathbf{x} = \mathbf{b}$ is the system:

$$3x_1 + 0x_2 = 3$$
$$1x_1 + 2x_2 = 4$$
$$0x_1 + 1x_2 = 1.5$$

### The Column View

Recall from Lecture 7 that we can also read matrix-vector multiplication column by column. If we write $A = [\mathbf{a}_1 \mid \mathbf{a}_2]$ where $\mathbf{a}_1 = \begin{bmatrix}3\\1\\0\end{bmatrix}$ and $\mathbf{a}_2 = \begin{bmatrix}0\\2\\1\end{bmatrix}$, then:

$$A\mathbf{x} = x_1\mathbf{a}_1 + x_2\mathbf{a}_2 = x_1\begin{bmatrix}3\\1\\0\end{bmatrix} + x_2\begin{bmatrix}0\\2\\1\end{bmatrix}$$

So $A\mathbf{x} = \mathbf{b}$ is asking: **can we find weights $x_1, x_2$ such that a linear combination of the columns of $A$ produces $\mathbf{b}$?**

From the first equation $x_1 = 1$, from the third $x_2 = 1.5$, and the second checks out: $1 + 2(1.5) = 4$ ✓. So $\mathbf{x} = \begin{bmatrix}1\\1.5\end{bmatrix}$ is the solution.

In [ ]:
A  = np.array([[3., 0.],
               [1., 2.],
               [0., 1.]])
b1 = np.array([[3.], [4.], [1.5]])
x  = np.array([[1.], [1.5]])

# Column vectors
a1 = A[:, 0:1]   # first column
a2 = A[:, 1:2]   # second column

print('Full matrix A:')
print(A)
print()
print('Column view: x1*a1 + x2*a2')
print(f'  a1 = {a1.flatten()},  a2 = {a2.flatten()}')
print(f'  1.0 * {a1.flatten()} + 1.5 * {a2.flatten()} = {(x[0,0]*a1 + x[1,0]*a2).flatten()}')
print()
print('Matrix multiply A @ x:')
print(A @ x)
print()
print('b1:', b1.T)
print('A @ x == b1:', np.allclose(A @ x, b1))

### Column Space and the Three Cases

**Definition.** The **column space** of $A$, written $C(A)$, is the set of all vectors that can be produced as linear combinations of the columns of $A$. The question "does $A\mathbf{x} = \mathbf{b}$ have a solution?" is equivalent to: **is $\mathbf{b}$ in $C(A)$?**

When we try to solve $A\mathbf{x} = \mathbf{b}$, exactly one of three things happens:

- **No solution:** $\mathbf{b}$ is not in $C(A)$. No weights produce $\mathbf{b}$ from the columns of $A$.
- **Unique solution:** $\mathbf{b}$ is in $C(A)$, and exactly one weight vector produces it.
- **Infinitely many solutions:** $\mathbf{b}$ is in $C(A)$, but multiple weight vectors produce it — the columns of $A$ are redundant.

**Definition.** The **rank** of $A$ is the number of linearly independent columns — the true dimensionality of $C(A)$. If $\text{rank}(A) = n$, $A$ is **full column rank**.

In [ ]:
# b2 is constructed by pushing b1 off C(A) along the direction perpendicular to the plane
n_hat = np.cross(a1.flatten(), a2.flatten())
n_hat = n_hat / np.linalg.norm(n_hat)
b2 = b1 + 3.0 * n_hat.reshape(-1, 1)   # NOT in C(A)

print(f'rank(A) = {np.linalg.matrix_rank(A)}')
print()
print('b1 (in C(A)):  ', b1.T)
print('b2 (not in C(A)):', np.round(b2, 4).T)
print()
print('A @ x == b1:', np.allclose(A @ x, b1))
print('A @ x == b2:', np.allclose(A @ x, b2))

In [ ]:
# Run this cell without modification
# C(A) is a 2D plane in R^3. b1 lies on it; b2 floats above with a perpendicular drop-line.
from matplotlib.lines import Line2D

a1_flat = a1.flatten();  a2_flat = a2.flatten()
b1_flat = b1.flatten();  b2_flat = b2.flatten()
b2_proj      = A @ np.linalg.lstsq(A, b2, rcond=None)[0]
b2_proj_flat = b2_proj.flatten()

fig = plt.figure(figsize=(8, 6.5))
ax  = fig.add_subplot(111, projection='3d')

s, t = np.linspace(-1.8, 1.8, 40), np.linspace(-1.8, 1.8, 40)
S, T = np.meshgrid(s, t)
ax.plot_surface(S*a1_flat[0]+T*a2_flat[0], S*a1_flat[1]+T*a2_flat[1],
                S*a1_flat[2]+T*a2_flat[2], alpha=0.28, color='steelblue', zorder=1)

ax.quiver(0,0,0, *a1_flat, color='steelblue', lw=2.2, arrow_length_ratio=0.14, zorder=3)
ax.quiver(0,0,0, *a2_flat, color='tomato',    lw=2.2, arrow_length_ratio=0.14, zorder=3)

ax.quiver(0,0,0, *b1_flat, color='seagreen', lw=2.5, arrow_length_ratio=0.10, zorder=5)
ax.scatter(*b1_flat, color='seagreen', s=90, zorder=7)
ax.text(b1_flat[0]+0.25, b1_flat[1]-0.15, b1_flat[2]+0.1,
        r'$\mathbf{b}_1$'+' (in $C(A)$)', color='seagreen', fontsize=10, fontweight='bold')

ax.quiver(0,0,0, *b2_flat, color='#444444', lw=2.2, arrow_length_ratio=0.10, zorder=5)
ax.scatter(*b2_flat, color='#444444', s=90, zorder=7)
ax.text(b2_flat[0]+0.2, b2_flat[1]-0.2, b2_flat[2]+0.1,
        r'$\mathbf{b}_2$'+' (not in $C(A)$)', color='#444444', fontsize=10, fontweight='bold')

ax.plot([b2_flat[0],b2_proj_flat[0]], [b2_flat[1],b2_proj_flat[1]],
        [b2_flat[2],b2_proj_flat[2]], color='#444444', lw=1.8, linestyle='--', alpha=0.85)
ax.scatter(*b2_proj_flat, color='#444444', s=55, marker='x', linewidths=2, zorder=7)

eps_dir = (b2_flat-b2_proj_flat)/np.linalg.norm(b2_flat-b2_proj_flat)*0.28
ip_dir  = a1_flat/np.linalg.norm(a1_flat)*0.28
c1,c2,c3 = b2_proj_flat+eps_dir, b2_proj_flat+eps_dir+ip_dir, b2_proj_flat+ip_dir
ax.plot([b2_proj_flat[0],c1[0],c2[0],c3[0],b2_proj_flat[0]],
        [b2_proj_flat[1],c1[1],c2[1],c3[1],b2_proj_flat[1]],
        [b2_proj_flat[2],c1[2],c2[2],c3[2],b2_proj_flat[2]],
        color='#444444', lw=1, alpha=0.7)

ax.view_init(elev=25, azim=340)
ax.set_xlabel(r'$x_1$', labelpad=2, fontsize=11)
ax.set_ylabel(r'$x_2$', labelpad=2, fontsize=11)
ax.set_zlabel(r'$x_3$', labelpad=2, fontsize=11)
ax.set_title(r'$C(A)$ is a plane in $\mathbb{R}^3$: $\mathbf{b}_1$ lies on it, $\mathbf{b}_2$ does not',
             fontsize=11, pad=10)
ax.legend(handles=[
    Line2D([0],[0], color='steelblue', lw=2.2, label=r'$\mathbf{a}_1, \mathbf{a}_2$ (columns of $A$)'),
    Line2D([0],[0], color='seagreen',  lw=2.5, label=r'$\mathbf{b}_1 \in C(A)$ — solution exists'),
    Line2D([0],[0], color='#444444',   lw=2.2, label=r'$\mathbf{b}_2 \notin C(A)$ — no exact solution'),
    Line2D([0],[0], color='#444444',   lw=1.8, linestyle='--', label=r'perpendicular distance to $C(A)$'),
], fontsize=8.5, loc='upper left', bbox_to_anchor=(-0.02, 1.0))
plt.tight_layout()
plt.show()

**Question.** The column space of this $3 \times 2$ matrix is a 2D plane inside $\mathbb{R}^3$. What would the column space of a $3 \times 3$ full-rank matrix look like? What about a rank-deficient $3 \times 2$ matrix whose two columns are scalar multiples of each other?

**Question.** The dashed line drops from $\mathbf{b}_2$ perpendicularly to the plane, with a right-angle marker at its foot. Keep this image in mind — it reappears in Section 9.4 when we derive the least squares solution.

In [ ]:
# Rank in practice: a rank-deficient matrix has redundant columns
A_deficient = np.array([[1., 2.],
                        [3., 6.],
                        [2., 4.]])

print(f'rank(A)           = {np.linalg.matrix_rank(A)}')
print(f'rank(A_deficient) = {np.linalg.matrix_rank(A_deficient)}')
print()
print('A_deficient col 1 = 2 * col 0 — only one independent direction.')
print('C(A_deficient) is a line, not a plane.')

**Question.** If the column space of `A_deficient` is a line rather than a plane, how many vectors $\mathbf{b} \in \mathbb{R}^3$ can be reached exactly by $A_{\text{deficient}}\mathbf{x} = \mathbf{b}$? What does this say about how useful a rank-deficient feature matrix would be for prediction?

## 9.2 The Matrix Inverse

The simplest case of $A\mathbf{x} = \mathbf{b}$ is when $A$ is square and full rank: $A \in \mathbb{R}^{n \times n}$ with $\text{rank}(A) = n$.

**Definition.** The **inverse** of a square matrix $A$ is the unique matrix $A^{-1}$ satisfying $A^{-1}A = AA^{-1} = I_n$. Think of it as undoing what $A$ does — just as $\frac{1}{a} \cdot a = 1$ in ordinary arithmetic.

$A^{-1}$ exists if and only if $A$ is square **and** full rank. A square full-rank matrix is called **invertible** or **nonsingular**; one that fails either condition is **singular**.

### Using $A^{-1}$ to solve $A\mathbf{x} = \mathbf{b}$

Multiply both sides on the left by $A^{-1}$:

$$A^{-1}A\mathbf{x} = A^{-1}\mathbf{b} \implies I\mathbf{x} = A^{-1}\mathbf{b} \implies \mathbf{x} = A^{-1}\mathbf{b}$$

When the inverse exists there is a **unique solution**, and $\mathbf{b}$ is guaranteed to be in $C(A)$.

In [ ]:
# Case 1: square, full rank — unique solution exists
A1    = np.array([[2., 1.],
                  [1., 2.]])
b1_sq = np.array([[3.], [3.]])

print(f'rank(A1) = {np.linalg.matrix_rank(A1)}')
x1 = np.linalg.inv(A1) @ b1_sq
print('Solution x1:', x1.T)
print('Verify A1 @ x1:', np.round(A1 @ x1, 4).T, '== b1:', b1_sq.T)

In [ ]:
# Case 2: square but rank-deficient — singular, no inverse
A2 = np.array([[4.,  2.],
               [-2., -1.]])

print(f'rank(A2) = {np.linalg.matrix_rank(A2)}')
print('Row 1 = -0.5 * Row 0 — collapses R^2 onto a line, cannot be inverted.')
print()
try:
    np.linalg.inv(A2)
except np.linalg.LinAlgError as e:
    print('LinAlgError:', e)

In [ ]:
# Case 3: non-square (tall, m > n) — full inverse cannot exist
A3 = np.array([[1., 2.],
               [3., 0.],
               [2., -3.]])

print(f'A3 shape: {A3.shape}  — non-square, no full inverse possible')
try:
    np.linalg.inv(A3)
except np.linalg.LinAlgError as e:
    print('LinAlgError:', e)
    print('We need a different approach.')

**Question.** Case 1 found $\mathbf{x} = \begin{bmatrix}1\\1\end{bmatrix}$. Verify by hand: does $2(1) + 1(1) = 3$? Does $1(1) + 2(1) = 3$?

**Question.** For `A2`, row 1 is $-\frac{1}{2}$ times row 0. What does this say about how many independent directions `A2` can point in? Why does this make inversion impossible?

## 9.3 Why Real Data Always Has No Exact Solution

The tall matrix case — $m > n$ — is exactly what arises when we have more data points than model parameters. With $m$ observations and $n$ weights to find, $A \in \mathbb{R}^{m \times n}$ with $m \gg n$.

$C(A)$ is an $n$-dimensional subspace inside $\mathbb{R}^m$. For a randomly chosen $\mathbf{b} \in \mathbb{R}^m$, the probability of landing exactly on that subspace is essentially zero. With real, noisy data, $\mathbf{b}$ will almost never be in $C(A)$.

To make this concrete, let's first see what the clean case looks like — synthetic data where an exact solution *does* exist — and then see what happens when we add real-world noise.

In [ ]:
# Clean data: exact solution exists (from Lecture 1)
data_clean = np.array([[2, 3,  8],    # web_3 = 1*web_1 + 2*web_2 exactly
                       [4, 6, 16],
                       [3, 7, 17],
                       [5, 2,  9],
                       [3, 2,  7]])

A_clean = data_clean[:, :2]
b_clean = data_clean[:, 2:3]

# With clean data we can verify: x = [1, 2] works
x_true = np.array([[1.], [2.]])
print('Clean data — A @ x_true:')
print((A_clean @ x_true).T)
print('b_clean:')
print(b_clean.T)
print('Exact solution exists:', np.allclose(A_clean @ x_true, b_clean))

In [ ]:
# Perturbed data: small noise breaks the exact solution
data_perturbed = np.array([[2, 3,  8],
                           [4, 6, 15],   # was 16, now 15
                           [3, 7, 18],   # was 17, now 18
                           [5, 2,  9],
                           [3, 2,  7]])

A = data_perturbed[:, :2]
b = data_perturbed[:, 2:3]

print('Perturbed data — A @ x_true (hoping to get b):')
print((A @ x_true).T)
print('b:')
print(b.T)
print()
print('Exact solution still exists:', np.allclose(A @ x_true, b))
print()
print('No exact solution. We need the least squares approach.')

**Question.** The two datasets differ by only a few values, yet an exact solution goes from achievable to impossible. Why? Think about what changed geometrically: the column space $C(A)$ is the same for both datasets, but $\mathbf{b}$ moved. Where did it move to?

## 9.4 The Least Squares Solution

We cannot solve $A\mathbf{x} = \mathbf{b}$ exactly. Instead we find the $\hat{\mathbf{x}}$ that produces the closest possible approximation — the $\hat{\mathbf{b}} = A\hat{\mathbf{x}}$ that minimizes $\|\mathbf{b} - A\mathbf{x}\|^2$.

Geometrically, $\hat{\mathbf{b}}$ is the point in $C(A)$ *closest* to $\mathbf{b}$ — its orthogonal projection onto the column space. This is the same idea as the vector projection from Homework 2, just extended from one direction to an entire subspace.

### Derivation

The error vector $\boldsymbol{\epsilon} = \mathbf{b} - \hat{\mathbf{b}} = \mathbf{b} - A\hat{\mathbf{x}}$ is minimized when it is **perpendicular to every column of $A$** — exactly the geometry shown in Section 9.1. Perpendicular to every column means:

$$A^T\boldsymbol{\epsilon} = \mathbf{0}$$

Substituting $\boldsymbol{\epsilon} = \mathbf{b} - A\hat{\mathbf{x}}$ and rearranging:

$$A^T(\mathbf{b} - A\hat{\mathbf{x}}) = \mathbf{0}$$
$$A^TA\hat{\mathbf{x}} = A^T\mathbf{b}$$

These are the **normal equations**. Now $A^TA$ is $n \times n$ and invertible as long as $A$ is full column rank. Multiplying both sides on the left by $(A^TA)^{-1}$:

$$\boxed{\hat{\mathbf{x}} = (A^TA)^{-1}A^T\mathbf{b}}$$

### Connection to the Ordinary Inverse

The matrix $(A^TA)^{-1}A^T$ is called the **left inverse** of $A$, and it satisfies:

$$(A^TA)^{-1}A^T \cdot A = I_n$$

When $A$ is square and invertible, $(A^TA)^{-1}A^T$ reduces to $A^{-1}$ exactly. The least squares formula is not a workaround — it is the natural generalization of the matrix inverse to the non-square case.

In [ ]:
# Verify the left-inverse property: (A^T A)^{-1} A^T A = I_n
left_inv = np.linalg.inv(A.T @ A) @ A.T
print('(A^T A)^{-1} A^T A  (should be I_2):')
print(np.round(left_inv @ A, 6))

## 9.5 The Objective Function and RMSE

Before computing the solution, let's pause on what we're actually minimizing — because this idea sits at the heart of supervised machine learning.

### The Objective Function

Least squares minimizes the **sum of squared errors** between the actual values $\mathbf{b}$ and the predicted values $\hat{\mathbf{b}} = A\hat{\mathbf{x}}$:

$$J(\mathbf{x}) = \|\mathbf{b} - A\mathbf{x}\|^2 = \sum_{i=1}^{m}(b_i - \hat{b}_i)^2$$

This is called an **objective function** — a single number that measures how bad our current weights are. The goal of training a supervised learning model is always to find the weights $\hat{\mathbf{x}}$ that minimize some objective function. Least squares is the simplest and most important example: the objective function is the squared error, and the formula $\hat{\mathbf{x}} = (A^TA)^{-1}A^T\mathbf{b}$ is its exact minimizer.

Every regression and classification algorithm you will encounter in this course and beyond is built around this same idea — defining a loss and finding the weights that minimize it.

### RMSE

The raw sum of squared errors $J$ grows with the number of data points, making it hard to compare across datasets. We prefer the **Root Mean Squared Error (RMSE)**, which puts the error back in the original units of $\mathbf{b}$:

$$\text{RMSE} = \sqrt{\frac{1}{m}\sum_{i=1}^{m}(b_i - \hat{b}_i)^2} = \frac{\|\mathbf{b} - \hat{\mathbf{b}}\|}{\sqrt{m}}$$

You first saw this in Homework 2 as the RMS of the error vector. Here $\mathbf{b}$ plays the role of the true values and $\hat{\mathbf{b}} = A\hat{\mathbf{x}}$ plays the role of the predictions. RMSE = 0 means a perfect fit; larger values mean the model is further from the data on average.

In [ ]:
# Compute the least squares solution for the perturbed data
x_hat = 

print('Least squares solution x_hat:')
print(np.round(x_hat, 4))
print()
print(f'Model: web_3_hat = {x_hat[0,0]:.4f} * web_1 + {x_hat[1,0]:.4f} * web_2')
print()
print('(True weights from Lecture 1 were x1=1.0, x2=2.0)')

In [ ]:
# Predicted values, objective function, and RMSE
b_hat   = 
epsilon = 
m       = 

J    = np.sum(epsilon**2)                    # objective function: sum of squared errors
rmse = np.linalg.norm(epsilon) / np.sqrt(m)  # RMSE = sqrt(J / m)

print('Actual b:   ', b.T)
print('Predicted:  ', np.round(b_hat, 4).T)
print('Errors e:   ', np.round(epsilon, 4).T)
print()
print(f'Objective J = sum of squared errors = {J:.4f}')
print(f'RMSE        = sqrt(J / m)           = {rmse:.4f}')
print()
# The defining property: epsilon is perpendicular to C(A)
print('A^T @ epsilon (should be ~0 — confirms we found the minimum):')
print(np.round(A.T @ epsilon, 10).T)

The fact that $A^T\boldsymbol{\epsilon} \approx \mathbf{0}$ is not just a check — it is the condition that *defines* the minimum of $J$. Any other $\mathbf{x}$ would leave a nonzero component in $\boldsymbol{\epsilon}$ that points along some column of $A$, meaning we could still improve the fit by adjusting $\mathbf{x}$. At $\hat{\mathbf{x}}$, no such improvement is possible.

**Question.** The least squares solution gives $\hat{\mathbf{x}} \approx (0.91, 2.07)$ rather than the true $(1.0, 2.0)$. We minimized $J$ perfectly — so why isn't $\hat{\mathbf{x}}$ equal to the true weights?

**Question.** If we doubled the number of data points (using the same underlying model plus noise), would you expect $J$ to increase, decrease, or stay the same? What about RMSE?

## 9.6 Application: Palmer Penguins

Let's apply the full pipeline to a real dataset. We'll predict `body_mass_g` from `flipper_length_mm` and `bill_depth_mm` — two features we know are correlated with mass from Homework 3.

In [ ]:
url = 'https://raw.githubusercontent.com/ajorgen1/Linear-Algebra-With-Applications/main/datasets/penguins.csv'
df  = pd.read_csv(url).dropna()

features   = ['flipper_length_mm', 'bill_length_mm', 'bill_depth_mm']
A_pen = df[features].to_numpy()
b_pen = df['body_mass_g'].to_numpy().reshape(-1, 1)

print(f'A shape: {A_pen.shape}   rank: {np.linalg.matrix_rank(A_pen)}')
print(f'b shape: {b_pen.shape}')

In [ ]:
# Least squares solution via the normal equations
x_hat_pen = np.linalg.inv(A_pen.T @ A_pen) @ A_pen.T @ b_pen

print('Weights x_hat:')
for name, w in zip(features, x_hat_pen.flatten()):
    print(f'  {name:<22}: {w:.4f}')
print()

b_hat_pen   = A_pen @ x_hat_pen
epsilon_pen = b_pen - b_hat_pen
m_pen       = b_pen.shape[0]

J_pen    = np.sum(epsilon_pen**2)
rmse_pen = np.linalg.norm(epsilon_pen) / np.sqrt(m_pen)

print(f'Objective J = {J_pen:.1f} g^2')
print(f'RMSE        = {rmse_pen:.1f} g')
print()
print('A^T @ epsilon (should be ~0):')
print(np.round(A_pen.T @ epsilon_pen, 4).T)

In [ ]:
# Run this cell without modification
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(b_pen, b_hat_pen, color='steelblue', alpha=0.6,
           edgecolors='white', linewidth=0.4, s=30)
lims = [b_pen.min() - 100, b_pen.max() + 100]
ax.plot(lims, lims, 'k--', lw=1, alpha=0.6, label='Perfect prediction')
ax.set_xlabel('Actual body mass (g)')
ax.set_ylabel('Predicted body mass (g)')
ax.set_title(r'Actual vs. Predicted: $\mathbf{b}$ vs. $\hat{\mathbf{b}}$')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)

ax = axes[1]
ax.hist(epsilon_pen.flatten(), bins=25, color='tomato', edgecolor='white', linewidth=0.4)
ax.axvline(0, color='black', lw=1, linestyle='--')
ax.set_xlabel(r'Residual $\epsilon_i = b_i - \hat{b}_i$ (g)')
ax.set_ylabel('Count')
ax.set_title(r'Distribution of Residuals $\boldsymbol{\epsilon}$')
ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle(f'Penguins Least Squares — RMSE = {rmse_pen:.1f} g', fontsize=12)
plt.tight_layout()
plt.show()

**Question.** Looking at the actual vs. predicted plot, do you notice any systematic patterns in where the model misses? Do you see clusters? What might explain them?

**Question.** The residual histogram is approximately centered at zero. Is this always true for any least squares solution, or does it depend on the data?

**Looking ahead.** Our model has no intercept term — it is forced through the origin, which is almost never appropriate for real data. In Lecture 10 we introduce the full regression framework: adding an intercept column to $A$, switching to the $\mathbf{X}\boldsymbol{\beta} = \mathbf{y}$ notation used in statistics and scikit-learn, and computing $R^2$ alongside RMSE to evaluate fit.